# Model Optimization: Knowledge Distillation

In this notebook, we'll apply knowledge distillation techniques to our models using distributed processing. Instead of running the distillation on our notebook instance, we'll launch separate SageMaker Processing jobs to perform the distillation on more powerful instances.

## What is Knowledge Distillation?

Knowledge distillation is a model compression technique where a smaller "student" model is trained to mimic the behavior of a larger "teacher" model. The key insight is that the teacher's outputs contain rich information beyond just the hard labels - they contain the teacher's "dark knowledge" about the relationships between classes and the confidence in predictions.

### Benefits of Knowledge Distillation:
- **Smaller Models**: Student models have fewer parameters and are much smaller in size
- **Faster Inference**: Smaller models perform inference more quickly
- **Lower Resource Requirements**: Students require less memory and compute
- **Preserved Accuracy**: Students can retain much of the teacher's accuracy

### How Knowledge Distillation Works:
1. **Teacher Outputs**: The teacher model produces "soft targets" (probability distributions)
2. **Temperature Scaling**: These distributions are softened using a temperature parameter
3. **Student Training**: The student is trained to match both the correct labels and the teacher's soft targets
4. **Knowledge Transfer**: The student learns not just the correct answers but the teacher's reasoning

### Distributed Processing Approach
This notebook uses SageMaker Processing jobs to perform knowledge distillation on separate, more powerful instances. This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel

## 1. Import Dependencies

In [ ]:
import os
import json
import torch
import time
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import boto3
import sagemaker
from sagemaker.processing import ProcessingInput, ProcessingOutput, Processor
from sagemaker.pytorch.processing import PyTorchProcessor

# Import our utility functions for distributed processing
from sagemaker_processing import run_distillation_job

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")
    
    # Set default values that user should update
    S3_BUCKET = "YOUR_BUCKET_NAME_HERE"  # Update this value
    AWS_REGION = "YOUR_REGION_HERE"      # Update this value
    SAGEMAKER_ROLE_ARN = "YOUR_ROLE_ARN_HERE"  # Update this value
    OPTIMIZATION_INSTANCE_TYPE = "ml.c5.xlarge"  # Default optimization instance type
    
    # Store the updated values
    %store S3_BUCKET
    %store AWS_REGION
    %store SAGEMAKER_ROLE_ARN
    %store OPTIMIZATION_INSTANCE_TYPE

## 3. Load Baseline Metrics and Model Information

In [ ]:
# Load baseline metrics from file
with open('baseline_metrics.json', 'r') as f:
    baseline_metrics = json.load(f)

print(f"Loaded baseline metrics for {len(baseline_metrics)} models")

# Load model information from file
with open('model_info.json', 'r') as f:
    model_info = json.load(f)

print(f"Loaded information for {len(model_info)} models")

## 4. Define Sample Inputs for Each Task

In [ ]:
# Define sample inputs for each task
sample_inputs = {
    "sentiment_analysis": "I really enjoyed this movie. The acting was superb and the plot was engaging.",
    "ner": "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington.",
    "question_answering": {
        "question": "What is machine learning?",
        "context": "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
    },
    "masked_lm": "The [MASK] is a large language model trained by OpenAI."
}

## 5. Examine and Upload Distillation Script to S3

In this section, we'll examine and upload the Python script that performs the actual knowledge distillation. This script will be executed on the SageMaker Processing instances.

### What the Script Does:
1. **Loads the teacher model and tokenizer** from Hugging Face
2. **Creates a smaller student model** with the same task capabilities
3. **Prepares a dataset** for training the student model
4. **Performs knowledge distillation** by training the student to mimic the teacher
5. **Measures performance metrics** like model size and inference time
6. **Saves the distilled model** and metrics to the output directory

### Key Components of Knowledge Distillation:
- **Temperature Parameter**: Controls how "soft" the teacher's probability distributions are
- **KL Divergence Loss**: Measures how well the student matches the teacher's distributions
- **Custom Trainer**: Implements the distillation loss function

The script uses DistilBERT as the student model architecture, which is specifically designed for knowledge distillation from BERT-based models.

In [ ]:
# Display the distillation script with syntax highlighting
!cat distillation_script.py

In [ ]:
# Upload the distillation script to S3
s3_client = boto3.client('s3')
s3_client.upload_file(
    'distillation_script.py', 
    S3_BUCKET, 
    'scripts/distillation_script.py'
)

print(f"Uploaded distillation script to s3://{S3_BUCKET}/scripts/distillation_script.py")

## 6. Launch Distributed Distillation Jobs

Now we'll set up and launch the SageMaker Processing jobs to perform knowledge distillation. Each model will be processed in a separate job, allowing for parallel processing.

### Distillation Process:
1. **Create a PyTorch processor** with the appropriate instance type and configuration
2. **For each model**:
   - Save and upload model information to S3
   - Define inputs (distillation script and model info) and outputs
   - Launch a processing job with the appropriate arguments
   - Store the job information for monitoring

We're using DistilBERT as the student model architecture, which is about 40% smaller than BERT while retaining about 97% of its language understanding capabilities. The distillation process involves training for 3 epochs with a batch size of 8.

In [ ]:
# Define the instance type to use for distillation
instance_type = OPTIMIZATION_INSTANCE_TYPE
print(f"Using instance type: {instance_type} for optimization jobs")

# Create a SageMaker session
sagemaker_session = sagemaker.Session()

# Create a PyTorch processor
processor = PyTorchProcessor(
    framework_version="1.13.1",
    py_version="py39",
    role=SAGEMAKER_ROLE_ARN,
    instance_type=instance_type,
    instance_count=1,
    base_job_name="model-distillation",
    sagemaker_session=sagemaker_session
)

In [ ]:
# Launch distillation jobs for each model
distillation_jobs = {}

for model_key in model_info.keys():
    # Skip models that are not suitable for distillation
    if model_info[model_key]['task'] not in ["sequence-classification", "token-classification", "question-answering", "masked-lm"]:
        print(f"Skipping {model_key}: task {model_info[model_key]['task']} not supported for distillation")
        continue
        
    print(f"\nLaunching distillation job for {model_key}...")
    
    # Save model info to a temporary file
    with open(f'temp_{model_key}_info.json', 'w') as f:
        json.dump({model_key: model_info[model_key]}, f)
    
    # Upload to S3
    s3_client.upload_file(
        f'temp_{model_key}_info.json', 
        S3_BUCKET, 
        f'optimization/inputs/{model_key}/model_info.json'
    )
    
    # Define inputs and outputs
    inputs = [
        ProcessingInput(
            source=f's3://{S3_BUCKET}/scripts/distillation_script.py',
            destination='/opt/ml/processing/input/code/distillation_script.py'
        ),
        ProcessingInput(
            source=f's3://{S3_BUCKET}/optimization/inputs/{model_key}/model_info.json',
            destination='/opt/ml/processing/input/data/model_info.json'
        )
    ]
    
    outputs = [
        ProcessingOutput(
            source='/opt/ml/processing/output',
            destination=f's3://{S3_BUCKET}/optimization/outputs/{model_key}_distilled'
        )
    ]
    
    # Run the processing job
    distillation_jobs[model_key] = processor.run(
        code='distillation_script.py',
        inputs=inputs,
        outputs=outputs,
        arguments=[
            '--model-info-path', '/opt/ml/processing/input/data/model_info.json',
            '--output-dir', '/opt/ml/processing/output',
            '--student-model-name', 'distilbert-base-uncased',
            '--num-epochs', '3',
            '--batch-size', '8'
        ]
    )
    
    print(f"Launched distillation job: {distillation_jobs[model_key].job_name}")

## 7. Monitor Job Status

After launching the distillation jobs, we need to monitor their progress. SageMaker Processing jobs run asynchronously, so we'll periodically check their status until all jobs are complete.

### Monitoring Process:
1. **Create a SageMaker client** to interact with the SageMaker API
2. **Check job status every 30 seconds** until all jobs are complete
3. **Display a status table** showing the current status of each job

Knowledge distillation is more time-consuming than quantization or pruning because it involves training the student model. Depending on the model size and instance type, this process can take from several minutes to a few hours.

In [ ]:
# Monitor job status
import time

# Create a SageMaker client
sagemaker_client = boto3.client('sagemaker')

# Check job status every 30 seconds
all_completed = False
while not all_completed:
    all_completed = True
    job_statuses = {}
    
    for model_key, job in distillation_jobs.items():
        response = sagemaker_client.describe_processing_job(
            ProcessingJobName=job.job_name
        )
        status = response['ProcessingJobStatus']
        job_statuses[model_key] = status
        
        if status in ['InProgress', 'Stopping']:
            all_completed = False
    
    # Display status table
    status_df = pd.DataFrame({
        'Model': list(job_statuses.keys()),
        'Status': list(job_statuses.values())
    })
    display(status_df)
    
    if not all_completed:
        print("Waiting for jobs to complete...")
        time.sleep(30)
    else:
        print("All jobs completed!")

## 8. Collect Results

Once all jobs are complete, we'll collect and combine the results from each job. Each job produces a metrics file containing information about the distilled model, such as size, inference time, and the comparison with the teacher model.

### Collection Process:
1. **Download metrics files** from S3 for each model
2. **Combine metrics** into a single dictionary
3. **Save combined metrics** to a local file for use in later notebooks

This gives us a comprehensive view of the distillation results across all models, which we'll analyze in the next section.

In [ ]:
# Download and combine results
distilled_metrics = {}

for model_key in distillation_jobs.keys():
    # Download metrics file
    try:
        s3_client.download_file(
            S3_BUCKET,
            f'optimization/outputs/{model_key}_distilled/distilled_metrics.json',
            f'temp_{model_key}_distilled_metrics.json'
        )
        
        # Load metrics
        with open(f'temp_{model_key}_distilled_metrics.json', 'r') as f:
            metrics = json.load(f)
        
        # Add to combined metrics
        distilled_metrics.update(metrics)
        
        print(f"Downloaded metrics for {model_key}")
    except Exception as e:
        print(f"Error downloading metrics for {model_key}: {e}")

# Save combined metrics
with open('distilled_metrics.json', 'w') as f:
    json.dump(distilled_metrics, f, indent=2)

print(f"\nSaved distilled metrics for {len(distilled_metrics)} models to distilled_metrics.json")

## 9. Compare Results

Now we'll compare the performance of the distilled student models against the original teacher models. This comparison helps us understand the impact of knowledge distillation on model size and inference speed.

### Key Metrics to Compare:
- **Model Size**: How much smaller are the student models?
- **Inference Time**: How much faster are the student models?
- **Size Reduction Percentage**: The percentage reduction in model size
- **Inference Speedup Percentage**: The percentage improvement in inference speed

We expect to see significant size reductions (typically 40-60%) and inference speedups (typically 30-50%) with knowledge distillation. The exact improvements depend on the specific teacher and student architectures.

In [ ]:
# Create a DataFrame for comparison
comparison_data = []

for model_key, metrics in distilled_metrics.items():
    teacher_key = metrics.get("teacher_model_key")
    if teacher_key and teacher_key in baseline_metrics:
        teacher = baseline_metrics[teacher_key]
        student = metrics
        
        # Calculate improvements
        size_reduction = (teacher['model_size'] - student['model_size']) / teacher['model_size'] * 100
        time_reduction = (teacher['inference_time'] - student['inference_time']) / teacher['inference_time'] * 100
        
        comparison_data.append({
            'Teacher Model': teacher['model_name'],
            'Student Model': student['model_name'],
            'Task': student['task'],
            'Teacher Size (MB)': teacher['model_size'],
            'Student Size (MB)': student['model_size'],
            'Size Reduction (%)': size_reduction,
            'Teacher Inference (ms)': teacher['inference_time'],
            'Student Inference (ms)': student['inference_time'],
            'Inference Speedup (%)': time_reduction,
            'Teacher Memory (MB)': teacher['memory_usage'],
            'Student Memory (MB)': student['memory_usage'],
            'Memory Reduction (%)': memory_reduction
        })

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Display the DataFrame
comparison_df

## 10. Deploy Models to SageMaker for Inference

Now that we've created distilled student models, let's deploy them to SageMaker endpoints for real-world inference testing. We'll deploy both the original teacher models and the distilled student models to compare their performance.

### Deployment Process:
1. **Create model artifacts** in S3 for both teacher and student models
2. **Create SageMaker models** using these artifacts
3. **Create endpoints** to host the models
4. **Wait for endpoint deployment** to complete

This will allow us to directly compare inference performance between the teacher and student models.

In [ ]:
# Create a SageMaker client
sagemaker_client = boto3.client('sagemaker')

# Select a model to deploy for testing
model_key = list(distilled_metrics.keys())[0]  # Just use the first model for testing
teacher_key = distilled_metrics[model_key].get("teacher_model_key")
print(f"Deploying model: {model_key} - {distilled_metrics[model_key]['model_name']}")
print(f"Teacher model: {teacher_key} - {baseline_metrics[teacher_key]['model_name']}")

# Create model artifacts in S3
teacher_model_data = f"s3://{S3_BUCKET}/models/{teacher_key}/original"
student_model_data = f"s3://{S3_BUCKET}/models/{model_key}/distilled"

# Create SageMaker model for teacher model
teacher_model_name = f"{teacher_key}-teacher"
sagemaker_client.create_model(
    ModelName=teacher_model_name,
    PrimaryContainer={
        'Image': '763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-inference:1.13.1-transformers4.26.0-cpu-py39-ubuntu20.04',
        'ModelDataUrl': teacher_model_data,
        'Environment': {
            'HF_MODEL_ID': baseline_metrics[teacher_key]['model_name'],
            'HF_TASK': baseline_metrics[teacher_key]['task']
        }
    },
    ExecutionRoleArn=SAGEMAKER_ROLE_ARN
)

# Create SageMaker model for student model
student_model_name = f"{model_key}-student"
sagemaker_client.create_model(
    ModelName=student_model_name,
    PrimaryContainer={
        'Image': '763104351884.dkr.ecr.us-east-1.amazonaws.com/huggingface-pytorch-inference:1.13.1-transformers4.26.0-cpu-py39-ubuntu20.04',
        'ModelDataUrl': student_model_data,
        'Environment': {
            'HF_MODEL_ID': f"distilled-{distilled_metrics[model_key]['model_name']}",
            'HF_TASK': distilled_metrics[model_key]['task']
        }
    },
    ExecutionRoleArn=SAGEMAKER_ROLE_ARN
)

# Create endpoint configurations
teacher_endpoint_config_name = f"{teacher_key}-teacher-config"
sagemaker_client.create_endpoint_config(
    EndpointConfigName=teacher_endpoint_config_name,
    ProductionVariants=[
        {
            'VariantName': 'AllTraffic',
            'ModelName': teacher_model_name,
            'InstanceType': 'ml.m5.large',
            'InitialInstanceCount': 1
        }
    ]
)

student_endpoint_config_name = f"{model_key}-student-config"
sagemaker_client.create_endpoint_config(
    EndpointConfigName=student_endpoint_config_name,
    ProductionVariants=[
        {
            'VariantName': 'AllTraffic',
            'ModelName': student_model_name,
            'InstanceType': 'ml.m5.large',
            'InitialInstanceCount': 1
        }
    ]
)

# Create endpoints
teacher_endpoint_name = f"{teacher_key}-teacher-endpoint"
sagemaker_client.create_endpoint(
    EndpointName=teacher_endpoint_name,
    EndpointConfigName=teacher_endpoint_config_name
)

student_endpoint_name = f"{model_key}-student-endpoint"
sagemaker_client.create_endpoint(
    EndpointName=student_endpoint_name,
    EndpointConfigName=student_endpoint_config_name
)

print(f"Deploying teacher endpoint: {teacher_endpoint_name}")
print(f"Deploying student endpoint: {student_endpoint_name}")
print("Deployment in progress... This may take several minutes.")

In [ ]:
# Wait for endpoints to be in service
def wait_for_endpoint(endpoint_name):
    status = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)['EndpointStatus']
    print(f"Endpoint {endpoint_name} status: {status}")
    
    while status == 'Creating':
        time.sleep(30)
        status = sagemaker_client.describe_endpoint(EndpointName=endpoint_name)['EndpointStatus']
        print(f"Endpoint {endpoint_name} status: {status}")
    
    return status

teacher_status = wait_for_endpoint(teacher_endpoint_name)
student_status = wait_for_endpoint(student_endpoint_name)

if teacher_status == 'InService' and student_status == 'InService':
    print("Both endpoints are ready for inference!")
else:
    print(f"Warning: Endpoints not in service. Teacher: {teacher_status}, Student: {student_status}")

## 11. Test Inference Performance

Now that our endpoints are deployed, let's test their inference performance. We'll send the same input to both endpoints and measure:
1. **Response time**: How long it takes to get a response
2. **Throughput**: How many requests can be processed per second
3. **Output quality**: Whether the outputs are similar between teacher and student models

This will give us a real-world comparison of the performance benefits of knowledge distillation.

In [ ]:
# Create a SageMaker runtime client for inference
runtime_client = boto3.client('sagemaker-runtime')

# Prepare input data based on the model task
task = distilled_metrics[model_key]['task']
if task == 'sequence-classification':
    input_data = {"inputs": sample_inputs["sentiment_analysis"]}
elif task == 'token-classification':
    input_data = {"inputs": sample_inputs["ner"]}
elif task == 'question-answering':
    input_data = {
        "inputs": {
            "question": sample_inputs["question_answering"]["question"],
            "context": sample_inputs["question_answering"]["context"]
        }
    }
elif task == 'masked-lm':
    input_data = {"inputs": sample_inputs["masked_lm"]}
else:
    input_data = {"inputs": "This is a test input."}

# Convert to JSON
payload = json.dumps(input_data)

# Function to measure inference time
def measure_endpoint_inference_time(endpoint_name, payload, num_runs=10):
    response_times = []
    responses = []
    
    for _ in range(num_runs):
        start_time = time.time()
        response = runtime_client.invoke_endpoint(
            EndpointName=endpoint_name,
            ContentType='application/json',
            Body=payload
        )
        end_time = time.time()
        
        response_time = (end_time - start_time) * 1000  # Convert to ms
        response_times.append(response_time)
        
        # Parse response
        response_body = response['Body'].read().decode('utf-8')
        responses.append(json.loads(response_body))
    
    avg_response_time = sum(response_times) / len(response_times)
    throughput = 1000 / avg_response_time  # Requests per second
    
    return {
        "avg_response_time": avg_response_time,
        "throughput": throughput,
        "responses": responses
    }

# Measure inference time for both endpoints
print("Testing teacher model endpoint...")
teacher_results = measure_endpoint_inference_time(teacher_endpoint_name, payload)

print("Testing student model endpoint...")
student_results = measure_endpoint_inference_time(student_endpoint_name, payload)

# Display results
print("\nInference Performance Results:")
print(f"Teacher Model - Avg Response Time: {teacher_results['avg_response_time']:.2f} ms, Throughput: {teacher_results['throughput']:.2f} req/s")
print(f"Student Model - Avg Response Time: {student_results['avg_response_time']:.2f} ms, Throughput: {student_results['throughput']:.2f} req/s")

# Calculate improvement
time_improvement = (teacher_results['avg_response_time'] - student_results['avg_response_time']) / teacher_results['avg_response_time'] * 100
throughput_improvement = (student_results['throughput'] - teacher_results['throughput']) / teacher_results['throughput'] * 100

print(f"\nImprovement with Knowledge Distillation:")
print(f"Response Time: {time_improvement:.2f}% faster")
print(f"Throughput: {throughput_improvement:.2f}% higher")

# Compare outputs
print("\nSample Output Comparison:")
print(f"Teacher Model Output: {teacher_results['responses'][0]}")
print(f"Student Model Output: {student_results['responses'][0]}")

## 12. Clean Up Endpoints

To avoid unnecessary costs, let's clean up the endpoints we created. SageMaker endpoints incur charges as long as they're running, so it's important to delete them when they're no longer needed.

In [ ]:
# Delete endpoints and endpoint configurations
print("Cleaning up endpoints...")

# Delete endpoints
sagemaker_client.delete_endpoint(EndpointName=teacher_endpoint_name)
sagemaker_client.delete_endpoint(EndpointName=student_endpoint_name)
print(f"Deleted endpoints: {teacher_endpoint_name}, {student_endpoint_name}")

# Delete endpoint configurations
sagemaker_client.delete_endpoint_config(EndpointConfigName=teacher_endpoint_config_name)
sagemaker_client.delete_endpoint_config(EndpointConfigName=student_endpoint_config_name)
print(f"Deleted endpoint configurations: {teacher_endpoint_config_name}, {student_endpoint_config_name}")

# Delete models
sagemaker_client.delete_model(ModelName=teacher_model_name)
sagemaker_client.delete_model(ModelName=student_model_name)
print(f"Deleted models: {teacher_model_name}, {student_model_name}")

print("\nCleanup complete!")

## 13. Compare All Optimization Techniques

Now that we've applied and tested all three optimization techniques (quantization, pruning, and knowledge distillation), let's compare their effectiveness.

### Comparison Metrics:
- **Model Size Reduction**: How much smaller is the optimized model?
- **Inference Speed Improvement**: How much faster is the optimized model?
- **Output Quality**: How similar are the outputs to the original model?
- **Implementation Complexity**: How difficult is it to implement the technique?
- **Training Requirements**: Does the technique require additional training?

This comparison will help us understand which optimization technique is best suited for different use cases.

In [ ]:
# Load metrics from all optimization techniques
try:
    with open('quantized_metrics.json', 'r') as f:
        quantized_metrics_all = json.load(f)
    print(f"Loaded quantized metrics for {len(quantized_metrics_all)} models")
except FileNotFoundError:
    quantized_metrics_all = {}
    print("No quantized metrics found")

try:
    with open('pruned_metrics.json', 'r') as f:
        pruned_metrics_all = json.load(f)
    print(f"Loaded pruned metrics for {len(pruned_metrics_all)} models")
except FileNotFoundError:
    pruned_metrics_all = {}
    print("No pruned metrics found")

# We already have distilled_metrics loaded
print(f"Loaded distilled metrics for {len(distilled_metrics)} models")

# Create a comparison table
comparison_data = []

# Find models that have been optimized with all three techniques
for model_key in baseline_metrics.keys():
    baseline = baseline_metrics[model_key]
    
    # Check if this model has been optimized with all techniques
    has_quantized = model_key in quantized_metrics_all
    has_pruned = model_key in pruned_metrics_all
    has_distilled = False
    distilled_key = None
    
    # Find the corresponding distilled model
    for dk, dv in distilled_metrics.items():
        if dv.get("teacher_model_key") == model_key:
            has_distilled = True
            distilled_key = dk
            break
    
    if has_quantized or has_pruned or has_distilled:
        # Add baseline data
        row = {
            'Model': baseline['model_name'],
            'Technique': 'Baseline',
            'Size (MB)': baseline['model_size'],
            'Inference Time (ms)': baseline['inference_time'],
            'Size Reduction (%)': 0,
            'Speed Improvement (%)': 0
        }
        comparison_data.append(row)
        
        # Add quantized data if available
        if has_quantized:
            quantized = quantized_metrics_all[model_key]
            size_reduction = (baseline['model_size'] - quantized['model_size']) / baseline['model_size'] * 100
            speed_improvement = (baseline['inference_time'] - quantized['inference_time']) / baseline['inference_time'] * 100
            
            row = {
                'Model': baseline['model_name'],
                'Technique': 'Quantization',
                'Size (MB)': quantized['model_size'],
                'Inference Time (ms)': quantized['inference_time'],
                'Size Reduction (%)': size_reduction,
                'Speed Improvement (%)': speed_improvement
            }
            comparison_data.append(row)
        
        # Add pruned data if available
        if has_pruned:
            pruned = pruned_metrics_all[model_key]
            size_reduction = (baseline['model_size'] - pruned['model_size']) / baseline['model_size'] * 100
            speed_improvement = (baseline['inference_time'] - pruned['inference_time']) / baseline['inference_time'] * 100
            
            row = {
                'Model': baseline['model_name'],
                'Technique': 'Pruning',
                'Size (MB)': pruned['model_size'],
                'Inference Time (ms)': pruned['inference_time'],
                'Size Reduction (%)': size_reduction,
                'Speed Improvement (%)': speed_improvement
            }
            comparison_data.append(row)
        
        # Add distilled data if available
        if has_distilled:
            distilled = distilled_metrics[distilled_key]
            size_reduction = (baseline['model_size'] - distilled['model_size']) / baseline['model_size'] * 100
            speed_improvement = (baseline['inference_time'] - distilled['inference_time']) / baseline['inference_time'] * 100
            
            row = {
                'Model': baseline['model_name'],
                'Technique': 'Knowledge Distillation',
                'Size (MB)': distilled['model_size'],
                'Inference Time (ms)': distilled['inference_time'],
                'Size Reduction (%)': size_reduction,
                'Speed Improvement (%)': speed_improvement
            }
            comparison_data.append(row)

# Create DataFrame
comparison_df = pd.DataFrame(comparison_data)

# Display the DataFrame
comparison_df

## 14. Visualization of Optimization Results

In [ ]:
# Create visualizations to compare optimization techniques
plt.figure(figsize=(12, 6))

# Group by technique and calculate mean values
technique_means = comparison_df.groupby('Technique').mean()

# Size reduction plot
plt.subplot(1, 2, 1)
sns.barplot(x=technique_means.index, y='Size Reduction (%)', data=technique_means)
plt.title('Average Model Size Reduction')
plt.ylabel('Size Reduction (%)')
plt.ylim(0, 100)

# Speed improvement plot
plt.subplot(1, 2, 2)
sns.barplot(x=technique_means.index, y='Speed Improvement (%)', data=technique_means)
plt.title('Average Inference Speed Improvement')
plt.ylabel('Speed Improvement (%)')
plt.ylim(0, 100)

plt.tight_layout()
plt.show()

## 15. Conclusion and Recommendations

Based on our experiments with quantization, pruning, and knowledge distillation, we can make the following recommendations:

### Quantization
- **Best for**: Quick optimization with minimal effort
- **Advantages**: No training required, easy to implement
- **Disadvantages**: Limited size reduction, potential accuracy impact
- **Recommended when**: You need a quick solution with minimal development effort

### Pruning
- **Best for**: Moderate optimization with some development effort
- **Advantages**: Good size reduction, minimal accuracy impact
- **Disadvantages**: Requires careful tuning of pruning parameters
- **Recommended when**: You need a balance between optimization and development effort

### Knowledge Distillation
- **Best for**: Maximum optimization with significant development effort
- **Advantages**: Greatest size reduction and speed improvement
- **Disadvantages**: Requires training, most complex to implement
- **Recommended when**: You need the most optimized model and can invest in training

### Combined Approach
For the best results, consider combining these techniques:
1. Start with knowledge distillation to create a smaller student model
2. Apply pruning to remove unnecessary weights
3. Finally, apply quantization to reduce precision

This combined approach can yield the greatest optimization benefits while maintaining acceptable accuracy.

## 16. Resource Cleanup

To avoid unnecessary costs, make sure all SageMaker resources have been cleaned up:
- Endpoints
- Endpoint configurations
- Models
- Processing jobs

You can use the AWS Management Console or the AWS CLI to verify that all resources have been properly deleted.